In [ ]:
# ----------------------------
# Setup the Jupyter version of Dash
# ----------------------------
from jupyter_dash import JupyterDash
JupyterDash.infer_jupyter_proxy_config()

import dash_leaflet as dl
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output
import plotly.express as px
import pandas as pd
import base64
import urllib.parse
import os

# ----------------------------
# CRUD Module import
# ----------------------------
from CRUD_Python_Module import AnimalShelter


# ----------------------------
# RESCUE CRITERIA CONFIGURATION
# Used by the scoring algorithm (Milestone 3 enhancement) — not for query construction.
# Enhancement v3: query construction has moved into the CRUD module. The dashboard
# still owns this dict for scoring purposes (primary_breeds, optimal age windows),
# but it no longer builds raw MongoDB query documents from it.
# ----------------------------
RESCUE_CRITERIA = {
    # Water Rescue: strong-swimming breeds; Intact Female; age cap 26 weeks.
    # Labrador Retriever Mix is primary — most proven water-rescue record.
    # Optimal 8-20 weeks targets peak early-socialization trainability window.
    "WR": {
        "animal_type": "Dog",
        "breeds": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"],
        "primary_breeds": ["Labrador Retriever Mix"],
        "sex": "Intact Female",
        "age_max_weeks": 26,
        "age_optimal_min": 8,
        "age_optimal_max": 20,
    },
    # Mountain/Wilderness Rescue: working/Nordic breeds; Intact Male; age cap 26 weeks.
    # German Shepherd and Siberian Husky Mix are primary — proven alpine performance.
    "MWR": {
        "animal_type": "Dog",
        "breeds": ["German Shepherd", "Alaskan Malamute", "Old English Sheepdog", "Siberian Husky Mix"],
        "primary_breeds": ["German Shepherd", "Siberian Husky Mix"],
        "sex": "Intact Male",
        "age_max_weeks": 26,
        "age_optimal_min": 8,
        "age_optimal_max": 20,
    },
    # Disaster/Individual Tracking: scent-tracking breeds; Intact Male; 20-300 weeks.
    # German Shepherd and Bloodhound are primary — gold-standard tracking breeds.
    # Optimal 52-156 weeks (1-3 years): experienced but still highly trainable.
    "DIT": {
        "animal_type": "Dog",
        "breeds": ["Doberman Pinscher", "German Shepherd", "Golden Retriever", "Bloodhound"],
        "primary_breeds": ["German Shepherd", "Bloodhound"],
        "sex": "Intact Male",
        "age_min_weeks": 20,
        "age_max_weeks": 300,
        "age_optimal_min": 52,
        "age_optimal_max": 156,
    },
}

# ----------------------------
# BREED INDEX — inverted index data structure (Milestone 3 enhancement)
# Maps each breed name to the set of rescue type codes it qualifies for.
# Built once at startup for O(1) breed-to-rescue-type lookup.
# ----------------------------
BREED_INDEX: dict = {}
for _rtype, _crit in RESCUE_CRITERIA.items():
    for _breed in _crit["breeds"]:
        BREED_INDEX.setdefault(_breed, set()).add(_rtype)


# ----------------------------
# Data / Model
# ----------------------------

# Credentials loaded from environment variables — not hardcoded.
# Set AAC_USERNAME and AAC_PASSWORD before running (e.g., export AAC_USERNAME=aacuser).
raw_username = os.environ.get("AAC_USERNAME")
raw_password = os.environ.get("AAC_PASSWORD")

if not raw_username or not raw_password:
    raise EnvironmentError(
        "Missing required environment variables: AAC_USERNAME and/or AAC_PASSWORD. "
        "Set them before running this notebook (e.g., export AAC_USERNAME=aacuser)."
    )

username = urllib.parse.quote_plus(raw_username)
password = urllib.parse.quote_plus(raw_password)

shelter = AnimalShelter(username, password)


def fetch_df(query: dict) -> pd.DataFrame:
    """Fetch records from Mongo via the generic read method and return a cleaned DataFrame."""
    try:
        records = shelter.read(query)
        dff = pd.DataFrame.from_records(records)
        if not dff.empty and "_id" in dff.columns:
            dff.drop(columns=["_id"], inplace=True)
        return dff
    except Exception as e:
        print(f"[fetch_df] Error reading documents: {e}")
        return pd.DataFrame()


# ----------------------------
# QUERY RESULT CACHE (Milestone 3 enhancement)
# Caches results by filter_type key to avoid redundant database round-trips.
# Enhancement v3: fetch_or_cache now calls shelter.get_rescue_candidates() for
# rescue types instead of building raw MongoDB query dicts in the notebook.
# ----------------------------
_query_cache: dict = {}


def fetch_or_cache(filter_type: str) -> pd.DataFrame:
    """Return cached results for filter_type, querying only on first call."""
    if filter_type in _query_cache:
        return _query_cache[filter_type].copy()

    if filter_type == "ALL":
        # Generic read — no rescue-specific criteria
        result = fetch_df({})
    else:
        # Enhancement v3: delegate to the CRUD module's named rescue method.
        # The dashboard no longer constructs raw MongoDB query documents for
        # rescue types — that responsibility belongs to the model layer.
        try:
            records = shelter.get_rescue_candidates(filter_type)
            result = pd.DataFrame.from_records(records)
            if not result.empty and "_id" in result.columns:
                result.drop(columns=["_id"], inplace=True)
        except (TypeError, ValueError) as e:
            # get_rescue_candidates raises these for invalid input — surface them clearly
            print(f"[fetch_or_cache] Invalid rescue type {filter_type!r}: {e}")
            result = pd.DataFrame()
        except Exception as e:
            print(f"[fetch_or_cache] Error fetching {filter_type!r}: {e}")
            result = pd.DataFrame()

    _query_cache[filter_type] = result
    return result.copy()


# ----------------------------
# SUITABILITY SCORING ALGORITHM (Milestone 3 enhancement)
# ----------------------------

def compute_suitability_score(row: pd.Series, criteria: dict) -> int:
    """
    Compute a weighted rescue suitability score for one animal row.

    Point weights:
      Breed primary match:    5 pts
      Breed secondary match:  3 pts
      Age in optimal window:  3 pts
      Sex exact match:        2 pts
    Maximum possible score:  10 pts
    """
    score = 0

    primary_set = set(criteria.get("primary_breeds", []))
    secondary_set = set(criteria["breeds"]) - primary_set
    breed = row.get("breed", "")

    if breed in primary_set:
        score += 5
    elif breed in secondary_set:
        score += 3

    try:
        age = float(row.get("age_upon_outcome_in_weeks", 0) or 0)
    except (TypeError, ValueError):
        age = 0.0

    opt_min = criteria.get("age_optimal_min", criteria.get("age_min_weeks", 0))
    opt_max = criteria.get("age_optimal_max", criteria.get("age_max_weeks", float("inf")))
    if opt_min <= age <= opt_max:
        score += 3

    if row.get("sex_upon_outcome", "") == criteria["sex"]:
        score += 2

    return score


def rank_candidates(dff: pd.DataFrame, criteria: dict) -> pd.DataFrame:
    """
    Score every row, add a 'rescue_score' column, and sort descending
    so the best candidates appear at the top of the table.
    """
    dff = dff.copy()
    dff["rescue_score"] = dff.apply(
        lambda row: compute_suitability_score(row, criteria), axis=1
    )
    return dff.sort_values("rescue_score", ascending=False).reset_index(drop=True)


# ----------------------------
# LIGHTWEIGHT STARTUP LOAD
# Enhancement v3: replaces the full 10,000-record startup query with a 100-record
# sample. The full dataset loads on demand when All Animals is first selected.
# This reduces startup time and memory usage proportionally to collection growth.
# ----------------------------
_startup_records = shelter.read_sample(100)
df = pd.DataFrame.from_records(_startup_records) if _startup_records else pd.DataFrame()
if not df.empty and "_id" in df.columns:
    df.drop(columns=["_id"], inplace=True)
print(f"Startup sample loaded: {len(df)} records (full dataset loads on first All Animals selection)")


# ----------------------------
# Helpers
# ----------------------------
def guess_lat_lon_columns(columns):
    """Try to detect latitude and longitude column names by matching common aliases."""
    cols = [c.lower() for c in columns]
    lat_candidates = ["location_lat", "lat", "latitude", "geo_lat", "y"]
    lon_candidates = ["location_long", "location_lng", "lon", "lng", "longitude", "geo_long", "geo_lng", "x"]

    lat_col = None
    lon_col = None

    for cand in lat_candidates:
        if cand in cols:
            lat_col = columns[cols.index(cand)]
            break

    for cand in lon_candidates:
        if cand in cols:
            lon_col = columns[cols.index(cand)]
            break

    return lat_col, lon_col


def build_logo_img(filename: str):
    """
    Robust logo loader: asserts existence, detects PNG vs JPEG by header bytes,
    encodes in Base64, and returns an html.Img with forced visibility.
    """
    assert os.path.exists(filename), f"Logo not found in CWD: {os.getcwd()} -> {filename}"

    with open(filename, "rb") as f:
        data = f.read()

    # PNG signature: 89 50 4E 47 0D 0A 1A 0A
    # JPEG signature: FF D8 FF
    if data[:8] == b"\x89PNG\r\n\x1a\n":
        mime = "image/png"
    elif data[:3] == b"\xff\xd8\xff":
        mime = "image/jpeg"
    else:
        mime = "image/png"

    encoded = base64.b64encode(data).decode("ascii")

    return html.Img(
        src=f"data:{mime};base64,{encoded}",
        style={
            "height": "110px",
            "display": "block",
            "margin": "0 auto",
            "border": "1px solid #ccc",
            "padding": "4px",
            "background": "white"
        },
    )


# ----------------------------
# CONTEXT-AWARE CHART CONFIGURATION (Milestone 3 enhancement)
# Maps each filter type to the most informative column and chart type.
# ----------------------------
_CHART_CONFIG = {
    "ALL": {"col": "breed",                     "chart": "pie",  "title": "Breed Distribution"},
    "WR":  {"col": "age_upon_outcome_in_weeks",  "chart": "hist", "title": "Age Distribution — Water Rescue Candidates (weeks)"},
    "MWR": {"col": "age_upon_outcome_in_weeks",  "chart": "hist", "title": "Age Distribution — Mountain/Wilderness Candidates (weeks)"},
    "DIT": {"col": "outcome_type",               "chart": "pie",  "title": "Outcome Type — Disaster/Tracking Candidates"},
}


# ----------------------------
# Dash App / Layout
# ----------------------------
app = JupyterDash(__name__)

logo_img = build_logo_img("Grazioso Salvare Logo.png")

app.layout = html.Div([
    html.Div([
        logo_img,
        html.Center(html.B(html.H1("SNHU CS-340 Dashboard"))),
        html.Center(html.P("Amanda Willbanks")),
    ]),

    html.Hr(),

    html.Div([
        html.H4("Filter Type"),
        dcc.RadioItems(
            id="filter-type",
            options=[
                {"label": "All animals", "value": "ALL"},
                {"label": "Water Rescue", "value": "WR"},
                {"label": "Mountain/Wilderness Rescue", "value": "MWR"},
                {"label": "Disaster/Individual Tracking", "value": "DIT"},
            ],
            value="ALL",
            labelStyle={"display": "block"},
        ),
    ], style={"padding": "10px"}),

    html.Hr(),

    # selected_rows=[0] ensures the first row is always selected on load
    # so the map and chart have data to render immediately.
    dash_table.DataTable(
        id="datatable-id",
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict("records"),

        page_size=10,
        sort_action="native",
        filter_action="native",
        row_selectable="single",
        selected_rows=[0],
        column_selectable="single",

        style_table={"height": "400px", "overflowY": "auto", "overflowX": "auto"},
        style_cell={"textAlign": "left", "fontFamily": "Arial", "fontSize": 12, "minWidth": "120px"},
    ),

    html.Br(),
    html.Hr(),

    html.Div(
        className="row",
        style={"display": "flex", "gap": "20px"},
        children=[
            html.Div(id="graph-id", className="col s12 m6", style={"flex": "1"}),
            html.Div(id="map-id", className="col s12 m6", style={"flex": "1"}),
        ],
    ),
])


# ----------------------------
# Callbacks
# ----------------------------

# 1) Filter -> update DataTable
# Enhancement v3: fetch_or_cache now delegates rescue queries to
# shelter.get_rescue_candidates() — the CRUD module owns all query logic.
@app.callback(
    Output("datatable-id", "data"),
    Output("datatable-id", "columns"),
    Input("filter-type", "value"),
)
def update_dashboard(filter_type):
    if filter_type != "ALL" and filter_type not in RESCUE_CRITERIA:
        raise ValueError(f"Unknown filter type: {filter_type!r}")

    dff = fetch_or_cache(filter_type)

    if dff.empty:
        return [], [{"name": i, "id": i} for i in df.columns]

    # Rank candidates when a rescue filter is active (Milestone 3 enhancement)
    if filter_type in RESCUE_CRITERIA:
        dff = rank_candidates(dff, RESCUE_CRITERIA[filter_type])

    return dff.to_dict("records"), [{"name": i, "id": i} for i in dff.columns]


# 2) Highlight selected column
@app.callback(
    Output("datatable-id", "style_data_conditional"),
    Input("datatable-id", "selected_columns"),
)
def update_styles(selected_columns):
    return [{
        "if": {"column_id": i},
        "background_color": "#D2F3FF"
    } for i in (selected_columns or [])]


# 3) Context-aware chart (Milestone 3 enhancement)
@app.callback(
    Output("graph-id", "children"),
    Input("datatable-id", "derived_virtual_data"),
    Input("filter-type", "value"),
)
def update_graphs(viewData, filter_type):
    if not viewData:
        return html.Div("No data to graph.")

    dff = pd.DataFrame(viewData)

    cfg = _CHART_CONFIG.get(filter_type, _CHART_CONFIG["ALL"])
    col = cfg["col"]
    chart_type = cfg["chart"]
    title = cfg["title"]

    if col not in dff.columns:
        col = "breed" if "breed" in dff.columns else dff.columns[0]
        chart_type = "pie"
        title = f"Top {col} (visible rows)"

    if chart_type == "hist":
        age_data = pd.to_numeric(dff[col], errors="coerce").dropna()
        fig = px.histogram(
            age_data,
            nbins=15,
            title=title,
            labels={"value": "Age (weeks)", "count": "Count"},
        )
        fig.update_layout(xaxis_title="Age (weeks)", yaxis_title="Count", showlegend=False)
    else:
        counts = dff[col].value_counts().head(10).reset_index()
        counts.columns = [col, "count"]
        fig = px.pie(counts, names=col, values="count", title=title)

    return dcc.Graph(figure=fig)


# 4) Map update
@app.callback(
    Output("map-id", "children"),
    Input("datatable-id", "derived_virtual_data"),
    Input("datatable-id", "derived_virtual_selected_rows"),
)
def update_map(viewData, index):
    if viewData is None or len(viewData) == 0:
        dff = df.copy()
    else:
        dff = pd.DataFrame.from_dict(viewData)

    row = 0 if not index else index[0]

    if dff.empty or row < 0 or row >= len(dff):
        return [html.Div("No data available to display on the map.")]

    lat_col, lon_col = guess_lat_lon_columns(dff.columns)

    try:
        if lat_col and lon_col:
            lat = float(dff.loc[row, lat_col])
            lon = float(dff.loc[row, lon_col])
        else:
            # Fallback to positional index only when named columns cannot be detected
            lat = float(dff.iloc[row, 13])
            lon = float(dff.iloc[row, 14])
    except Exception:
        return [html.Div("Could not determine valid latitude/longitude for selected row.")]

    # Named column lookups — not magic integer indexes.
    TOOLTIP_COL = "breed"
    NAME_COL = "name"

    tooltip_text = str(dff.loc[row, TOOLTIP_COL]) if TOOLTIP_COL in dff.columns else "Animal"
    animal_name = str(dff.loc[row, NAME_COL]) if NAME_COL in dff.columns else "(no name)"

    # Map centered on Austin, TX (30.75, -97.48) — the Austin Animal Center location.
    return [
        dl.Map(
            style={"width": "100%", "height": "500px"},
            center=[30.75, -97.48],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),
                dl.Marker(
                    position=[lat, lon],
                    children=[
                        dl.Tooltip(tooltip_text),
                        dl.Popup([
                            html.H1("Animal Name"),
                            html.P(animal_name)
                        ])
                    ],
                ),
            ],
        )
    ]


# ----------------------------
# Run
# ----------------------------
app.run_server(debug=True, port=8052)
